In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import threading
import queue
import socket

# Initialize MediaPipe Hands
LAPTOP_PORT = 12345
ESP32_IP = "111.111.111.11"
result_queue = queue.Queue()
mp_hands = mp.solutions.hands
gesture="empty"
command="empty"
hands = mp_hands.Hands(static_image_mode=False,
                       max_num_hands=2,
                       min_detection_confidence=0.5,
                       min_tracking_confidence=0.5)

# Drawing utilities
mp_drawing = mp.solutions.drawing_utils
url="111.111.111.11"

# Start video capture
cap = cv2.VideoCapture(url)

gesture_triggers = {
    'OK': {'Left': False, 'Right': False},
    'Rock': {'Left': False, 'Right': False},
    'Point' :{'Left': False, 'Right': False},
    'Live'  :{'Left': False, 'Right': False},
    'Fist' :{'Left': False, 'Right': False}
}

def is_fist(landmarks,label= None):
    # Each finger: tip below (or close to) the MCP joint
    folded_fingers = 0

    fingers = [
        (4, 2),   # Thumb: tip vs MCP (approximated using landmark 2)
        (8, 5),   # Index: tip vs MCP
        (12, 9),  # Middle: tip vs MCP
        (16, 13), # Ring: tip vs MCP
        (20, 17), # Pinky: tip vs MCP
    ]

    for tip_idx, base_idx in fingers:
        tip = landmarks[tip_idx]
        base = landmarks[base_idx]
        if tip.y > base.y:  # Tip below base = folded
            folded_fingers += 1

    return folded_fingers >= 4  # At least 4 folded means a fist


def is_live_long(landmarks,label= None):
    # Get key finger tip and base joints
    index_tip = landmarks[8]
    index_mcp = landmarks[5]

    middle_tip = landmarks[12]
    middle_mcp = landmarks[9]

    ring_tip = landmarks[16]
    ring_mcp = landmarks[13]

    pinky_tip = landmarks[20]
    pinky_mcp = landmarks[17]

    # Compute distances between fingertips (x-axis separation)
    def x_dist(a, b):
        return abs(a.x - b.x)

    # Criteria:
    close_index_middle = x_dist(index_tip, middle_tip) < 0.05
    close_ring_pinky = x_dist(ring_tip, pinky_tip) < 0.05
    wide_middle_ring = x_dist(middle_tip, ring_tip) > 0.07

    return close_index_middle and close_ring_pinky and wide_middle_ring

def is_ok_sign(landmarks, label=None):
    import numpy as np

    # Distance between thumb tip and index tip
    thumb_index_dist = np.linalg.norm(np.array([
        landmarks[4].x - landmarks[8].x,
        landmarks[4].y - landmarks[8].y,
        landmarks[4].z - landmarks[8].z
    ]))

    # Check that middle, ring, and pinky are extended (tip above pip)
    middle_extended = landmarks[12].y < landmarks[10].y
    ring_extended   = landmarks[16].y < landmarks[14].y
    pinky_extended  = landmarks[20].y < landmarks[18].y

    # OK sign = fingers extended + thumb touching index
    is_fingers_extended = middle_extended and ring_extended and pinky_extended

    return thumb_index_dist < 0.05 and is_fingers_extended


def is_pointing_hand(landmarks, label):
    index_tip = landmarks[8]
    index_pip = landmarks[6]

    middle_tip = landmarks[12]
    middle_pip = landmarks[10]

    ring_tip = landmarks[16]
    ring_pip = landmarks[14]

    pinky_tip = landmarks[20]
    pinky_pip = landmarks[18]

    thumb_tip = landmarks[4]
    thumb_ip = landmarks[3]

    # Index finger extended (horizontally outward from hand center)
    if label == 'Left':
        index_extended = index_tip.x > index_pip.x
    else:  # Left
        index_extended = index_tip.x < index_pip.x

    # Other fingers folded (stay close to palm in x-axis)
    if label == 'Left':
        middle_folded = middle_tip.x < middle_pip.x
        ring_folded = ring_tip.x < ring_pip.x
        pinky_folded = pinky_tip.x < pinky_pip.x
    else:
        middle_folded = middle_tip.x > middle_pip.x
        ring_folded = ring_tip.x > ring_pip.x
        pinky_folded = pinky_tip.x > pinky_pip.x

    # Thumb check should still use y-axis (it folds vertically)
    thumb_folded = thumb_tip.y < thumb_ip.y

    return index_extended and middle_folded and ring_folded and pinky_folded and thumb_folded



def is_rock_on(landmarks,label=None):
    # Tip and PIP landmarks for fingers
    def is_extended(tip_idx, pip_idx):
        return landmarks[tip_idx].y < landmarks[pip_idx].y

    def is_folded(tip_idx, pip_idx):
        return landmarks[tip_idx].y > landmarks[pip_idx].y

    index_extended = is_extended(8, 6)
    pinky_extended = is_extended(20, 18)
    middle_folded = is_folded(12, 10)
    ring_folded = is_folded(16, 14)

    return index_extended and pinky_extended and middle_folded and ring_folded


def check_the_hand(sign_func, label, gesture_name, hand_landmarks):
    global gesture
    global command
    if sign_func(hand_landmarks.landmark,label):
        if not gesture_triggers[gesture_name][label]:
            gesture = gesture_name
            if label == 'Left':
                if gesture_name == 'OK':
                    result_queue.put('W')
                    command = "move forward"
                elif gesture_name == 'Rock':
                    result_queue.put('P')
                    command = "move right"
                elif gesture_name == 'Fist':
                    result_queue.put('E')
                    command = "light on"
            elif label == 'Right':
                if gesture_name == 'OK':
                    result_queue.put('Q')
                    command = "Stop"
                elif gesture_name == 'Rock':
                    result_queue.put('A')
                    command = "go left"
                elif gesture_name == 'Fist':
                    result_queue.put('R')
                    command = "light off"
            print(f"{label} hand: {gesture_name} sign command is sent")
            gesture_triggers[gesture_name][label] = True
        

    else:
        gesture_triggers[gesture_name][label] = False
        # gesture="empty"
                  


def is_thumbs_up(landmarks):
    # Thumb landmarks
    thumb_tip = landmarks[4]
    thumb_ip = landmarks[3]
    thumb_mcp = landmarks[2]

    # Other fingers (index, middle, ring, pinky)
    folded_fingers = True
    for tip_idx, pip_idx in [(8, 6), (12, 10), (16, 14), (20, 18)]:
        if landmarks[tip_idx].y < landmarks[pip_idx].y:
            folded_fingers = False
            break

    # Thumb extended upward
    thumb_up = thumb_tip.y < thumb_ip.y < thumb_mcp.y


    return thumb_up and folded_fingers

def gesture_thread():
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        # Flip image for natural visualization and convert to RGB
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
        # Process the frame and detect hands
        results = hands.process(rgb_frame)
    
        if results.multi_hand_landmarks and results.multi_handedness:
            for i, (hand_landmarks, hand_handedness) in enumerate(
                    zip(results.multi_hand_landmarks, results.multi_handedness)):
    
                label = hand_handedness.classification[0].label  # 'Left' or 'Right'
                score = hand_handedness.classification[0].score
    
                # Number of landmarks (usually 21)
                num_points = len(hand_landmarks.landmark)
    
                #print(f"Hand {i}: {label} (confidence: {score:.2f}), Landmarks: {num_points}")
    
                # Draw keypoints and connections
                mp_drawing.draw_landmarks(
                    frame, hand_landmarks, mp_hands.HAND_CONNECTIONS,)
    
                h, w, _ = frame.shape
                for idx, landmark in enumerate(hand_landmarks.landmark):
                    cx, cy = int(landmark.x * w), int(landmark.y * h)
                    cv2.putText(frame, str(idx), (cx, cy),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1, cv2.LINE_AA)
    
                # Optionally write hand info on screen
                wrist_x, wrist_y = int(hand_landmarks.landmark[0].x * w), int(hand_landmarks.landmark[0].y * h)
                # hand_info = f"Hand {i}: {label} ({score:.2f})"
                # cv2.putText(frame, hand_info, (wrist_x, wrist_y - 10),
                #             cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2, cv2.LINE_AA)
                
                
                
    
                check_the_hand(is_ok_sign, label, 'OK', hand_landmarks)
                check_the_hand(is_rock_on, label, 'Rock', hand_landmarks)
                
                check_the_hand(is_fist,label,'Fist',hand_landmarks)
                cv2.putText(frame, f"Gesture: {gesture} : {command}", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
                
                
                
                
        # Display result
        cv2.imshow('MediaPipe Hands', frame)
        if cv2.waitKey(1) & 0xFF == 27:  # Press ESC to exit
            cap.release()
            cv2.destroyAllWindows()
            break

def send_commands():
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.connect((ESP32_IP, LAPTOP_PORT))
            print(f"Connected to {ESP32_IP}:{LAPTOP_PORT}")
        
            while True:
                
                try:
                    command= result_queue.get()
                    if command is None:
                        break
                    s.sendall(command.encode()+b'\n')
                    print(f"Sent {command} to ESP32-CAM")
                    
                except Exception as e:
                    print(f"Error: {e}")
                    break

thread_receive_frames = threading.Thread(target=gesture_thread) # control the thread in the background 
thread_send_commands = threading.Thread(target=send_commands)

thread_receive_frames.start()
thread_send_commands.start()

thread_receive_frames.join()
thread_send_commands.join()


Connected to 192.168.1.106:12345
Left hand: Fist sign command is sentSent E to ESP32-CAM

Left hand: Fist sign command is sentSent E to ESP32-CAM

Left hand: Fist sign command is sentSent E to ESP32-CAM

Left hand: Fist sign command is sentSent E to ESP32-CAM

Right hand: Fist sign command is sentSent R to ESP32-CAM

Left hand: Fist sign command is sentSent E to ESP32-CAM

Left hand: Fist sign command is sentSent E to ESP32-CAM

Left hand: Fist sign command is sentSent E to ESP32-CAM

Left hand: Fist sign command is sent
Sent E to ESP32-CAM
Left hand: Fist sign command is sentSent E to ESP32-CAM

Left hand: Fist sign command is sent
Sent E to ESP32-CAM
Left hand: Fist sign command is sentSent E to ESP32-CAM

Left hand: Fist sign command is sent
Sent E to ESP32-CAM
Left hand: Fist sign command is sentSent E to ESP32-CAM

Left hand: Fist sign command is sent
Sent E to ESP32-CAM
Left hand: Fist sign command is sentSent E to ESP32-CAM

Left hand: Fist sign command is sent
Sent E to ESP32-C